# 16 — Package & Distribute

Takes the best model produced by the training pipeline and packages it for distribution.

## What this notebook does

```
Best fused model (notebook 15)  —OR—  warm-start adapter (notebook 04)
         ↓
  1. Verify & select best checkpoint
         ↓
  2. Quantize to 4-bit MLX  →  data/release/aro-coder-4bit/
         ↓
  3. Smoke-test the quantized model
         ↓
  4a. MLX server  →  startup script  (local OpenAI-compatible API)
  4b. Ollama      →  Modelfile + aro-coder:latest  (local chat)
  4c. HuggingFace →  push to mlx-community/aro-coder-4bit  (public)
```

Each distribution target (4a / 4b / 4c) is independent — run only the ones you need.

**Prerequisites:** notebook 04 (minimum) or notebook 15 (best quality)
**Install:** `pip install mlx-lm huggingface_hub`

## Setup

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')

import json, re, subprocess, shutil, time

MODELS_DIR  = GLOBAL_OUT_DIR / '../models'
RELEASE_DIR = GLOBAL_OUT_DIR / '../release'
RELEASE_DIR.mkdir(parents=True, exist_ok=True)

QUANT_DIR   = RELEASE_DIR / 'aro-coder-4bit'

# HuggingFace repo (change to your org/username)
HF_REPO_ID  = 'ARO-Lang/aro-coder-4bit'

# Ollama model name
OLLAMA_NAME = 'aro-coder'

print(f'Models dir:  {MODELS_DIR}')
print(f'Release dir: {RELEASE_DIR}')
print(f'HF repo:     {HF_REPO_ID}')
print(f'Ollama name: {OLLAMA_NAME}')

## Step 1 — Select best checkpoint

Looks for the highest-numbered fused round from notebook 13.
Falls back to the warm-start adapter fused with the base model if no rounds exist.

In [ ]:
def find_best_fused_model():
    """Return path to the best available fused model directory."""

    # Prefer the highest completed round from notebook 13
    if MODELS_DIR.exists():
        rounds = sorted(
            [d for d in MODELS_DIR.glob('round_*/fused') if (d / 'config.json').exists()],
            key=lambda p: int(re.search(r'round_(\d+)', str(p)).group(1))
        )
        if rounds:
            best = rounds[-1]
            round_n = re.search(r'round_(\d+)', str(best)).group(1)
            print(f'Found fused model: round {round_n} → {best}')
            return best, f'round_{round_n}_fused'

    # Fall back: fuse warm-start adapter on the fly
    warm = resolve_warm_adapter()
    if warm:
        fused_path = RELEASE_DIR / 'warm_start_fused'
        if not (fused_path / 'config.json').exists():
            print(f'No fused rounds found — fusing warm-start adapter...')
            cmd = [
                sys.executable, '-m', 'mlx_lm', 'fuse',
                '--model',        MODEL_ID,
                '--adapter-path', warm,
                '--save-path',    str(fused_path),
            ]
            subprocess.run(cmd, check=True)
        else:
            print(f'Using cached warm-start fused model: {fused_path}')
        return fused_path, 'warm_start_fused'

    # Last resort: use the base model directly (no fine-tuning done yet)
    print(f'WARNING: No fine-tuned model found — will quantize the base model.')
    print(f'Run notebook 04 (warm-start) or 13 (iterative) before packaging.')
    return MODEL_ID, 'base'


SOURCE_MODEL, SOURCE_LABEL = find_best_fused_model()
print(f'\nSource model : {SOURCE_MODEL}')
print(f'Source label : {SOURCE_LABEL}')

## Step 2 — Quantize to 4-bit MLX

In [ ]:
def quantize(source, dest, q_bits=4, q_group_size=64):
    """Quantize source model to q_bits and save to dest."""
    dest = Path(dest)
    if (dest / 'config.json').exists():
        print(f'Quantized model already exists at {dest} — skipping.')
        print('Delete the directory to re-quantize.')
        return dest

    dest.mkdir(parents=True, exist_ok=True)
    print(f'Quantizing {source} → {dest} ({q_bits}-bit, group={q_group_size})...')

    cmd = [
        sys.executable, '-m', 'mlx_lm', 'convert',
        '--hf-path',      str(source),
        '--mlx-path',     str(dest),
        '--quantize',
        '--q-bits',       str(q_bits),
        '--q-group-size', str(q_group_size),
    ]
    t0 = time.time()
    result = subprocess.run(cmd)
    elapsed = time.time() - t0

    if result.returncode != 0:
        shutil.rmtree(dest, ignore_errors=True)
        raise RuntimeError('Quantization failed.')

    size_gb = sum(f.stat().st_size for f in dest.rglob('*') if f.is_file()) / 1e9
    print(f'Done in {elapsed:.0f}s  |  size: {size_gb:.1f} GB  |  saved to {dest}')
    return dest


QUANT_DIR = quantize(SOURCE_MODEL, QUANT_DIR)
print(f'\nQuantized model ready: {QUANT_DIR}')

## Step 3 — Smoke test

Load the quantized model and run a quick ARO generation to verify it works.

In [ ]:
kb = load_knowledge()
SYSTEM_PROMPT = build_system_prompt(kb)

load_fn, generate_fn, make_sampler_fn = ensure_mlx_lm()

print(f'Loading quantized model from {QUANT_DIR}...')
model, tokenizer = load_fn(str(QUANT_DIR))
print('Model loaded.\n')

TEST_PROMPTS = [
    'Write an ARO feature set that retrieves a user by ID from a repository and returns it as an OK response.',
    'How do I emit a custom event in ARO and handle it in another feature set?',
    'Write a minimal ARO Application-Start that starts an HTTP server.',
]

for prompt in TEST_PROMPTS:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    output = generate_fn(
        model, tokenizer,
        prompt=tokens,
        max_tokens=400,
        sampler=make_sampler_fn(temp=0.2),
        verbose=False,
    )
    print(f'PROMPT: {prompt}')
    print(f'RESPONSE:\n{output.strip()}')
    print('─' * 70)

del model
print('\nSmoke test complete.')

## Step 4a — MLX server (local OpenAI-compatible API)

Writes a `serve.sh` startup script. Run it to expose the model on `http://localhost:8080/v1`
with the ARO system prompt pre-loaded.

Compatible with any OpenAI client: `curl`, Python `openai` SDK, VS Code Copilot Chat, etc.

In [ ]:
SERVE_SCRIPT = RELEASE_DIR / 'serve.sh'
CHAT_TEMPLATE_FILE = RELEASE_DIR / 'aro_system_prompt.txt'

# Write the system prompt to a standalone file
CHAT_TEMPLATE_FILE.write_text(SYSTEM_PROMPT)

# Write the startup script
serve_content = f"""#!/usr/bin/env bash
# ARO Coder — local MLX server
# Exposes an OpenAI-compatible API at http://localhost:8080/v1
#
# Usage:
#   ./serve.sh
#   curl http://localhost:8080/v1/chat/completions \\
#        -H 'Content-Type: application/json' \\
#        -d '{{"model":"aro-coder","messages":[{{"role":"user","content":"Write hello world in ARO"}}]}}'

MODEL="{QUANT_DIR}"
PORT=8080

python -m mlx_lm.server \\
    --model "$MODEL" \\
    --port  "$PORT" \\
    --system-prompt "$(cat {CHAT_TEMPLATE_FILE})"
"""
SERVE_SCRIPT.write_text(serve_content)
SERVE_SCRIPT.chmod(0o755)

print(f'MLX server script: {SERVE_SCRIPT}')
print(f'System prompt:     {CHAT_TEMPLATE_FILE}')
print()
print('To start the server:')
print(f'  {SERVE_SCRIPT}')
print()
print('To test (in another terminal):')
print("  curl http://localhost:8080/v1/chat/completions \\")
print("       -H 'Content-Type: application/json' \\")
print('       -d \'{"model":"aro-coder","messages":[{"role":"user","content":"Write hello world in ARO"}]}\'')

## Step 4b — Ollama

Creates an Ollama `Modelfile` and registers `aro-coder:latest` locally.
Requires [Ollama](https://ollama.com) to be installed and running.

Ollama uses GGUF format internally. The Modelfile points to the quantized MLX weights
via the `mlx` provider (Ollama ≥ 0.4 supports MLX models on Apple Silicon directly).

In [ ]:
MODELFILE_PATH = RELEASE_DIR / 'Modelfile'

# Escape system prompt for Modelfile (triple-quote safe)
system_escaped = SYSTEM_PROMPT.replace('"""', '\\"\\"\\"')

modelfile_content = f"""# ARO Coder — Ollama Modelfile
# Generated by notebook 14
#
# Build:  ollama create {OLLAMA_NAME} -f {MODELFILE_PATH}
# Run:    ollama run {OLLAMA_NAME}
# API:    curl http://localhost:11434/api/generate -d '{{"model":"{OLLAMA_NAME}","prompt":"Write hello world in ARO"}}'

FROM {QUANT_DIR}

SYSTEM """{system_escaped}"""

PARAMETER temperature    0.3
PARAMETER top_p          0.9
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx        4096

MESSAGE user      "Write a minimal ARO hello world application."
MESSAGE assistant "(Application-Start: Hello World) {{\\n    Log \\"Hello, World!\\" to the <console>.\\n    Return an <OK: status> for the <startup>.\\n}}"
"""

MODELFILE_PATH.write_text(modelfile_content)
print(f'Modelfile written: {MODELFILE_PATH}')

# Check if Ollama is available
ollama_bin = shutil.which('ollama')
if ollama_bin:
    print(f'\nOllama found: {ollama_bin}')
    print(f'Creating {OLLAMA_NAME}:latest ...')
    result = subprocess.run(
        ['ollama', 'create', OLLAMA_NAME, '-f', str(MODELFILE_PATH)],
        text=True
    )
    if result.returncode == 0:
        print(f'\n✓ Model registered. Run it with:')
        print(f'  ollama run {OLLAMA_NAME}')
    else:
        print('Ollama create failed — check Ollama is running (ollama serve)')
else:
    print('\nOllama not found in PATH.')
    print('Install from https://ollama.com, then run:')
    print(f'  ollama create {OLLAMA_NAME} -f {MODELFILE_PATH}')

## Step 4c — Push to HuggingFace

Uploads the quantized model to `HF_REPO_ID` on HuggingFace.
Requires `huggingface-cli login` (run once in terminal before this cell).

Set `HF_REPO_ID` at the top of this notebook to your org/username.

In [ ]:
try:
    from huggingface_hub import HfApi, login
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    from huggingface_hub import HfApi, login

def push_to_hf(local_dir, repo_id, private=False):
    api = HfApi()

    # Create repo if it doesn't exist
    try:
        api.create_repo(repo_id=repo_id, repo_type='model', private=private, exist_ok=True)
        print(f'Repo ready: https://huggingface.co/{repo_id}')
    except Exception as e:
        print(f'Could not create repo: {e}')
        print('Make sure you are logged in: huggingface-cli login')
        return

    # Write a README
    readme = f"""---
language: en
license: apache-2.0
tags:
  - aro
  - code-generation
  - mlx
  - 4-bit
base_model: {MODEL_ID}
---

# ARO Coder (4-bit MLX)

Fine-tuned version of [{MODEL_ID}](https://huggingface.co/{MODEL_ID})
specialised in the [ARO programming language](https://github.com/arolang/aro).

## Usage

```python
from mlx_lm import load, generate
model, tokenizer = load("{repo_id}")
response = generate(model, tokenizer, prompt="Write a user API in ARO", max_tokens=500)
print(response)
```

## Training

Trained with the ARO training pipeline (notebooks 01–13) on:
- 92 runnable ARO examples with execution-based validation
- 35 wiki pages (instruction/answer pairs)
- ARO language book Q&A pairs
- Actions reference pairs (all 60 built-in actions)
- Iterative self-improvement rounds
"""
    (Path(local_dir) / 'README.md').write_text(readme)

    print(f'Uploading {local_dir} → {repo_id} ...')
    api.upload_folder(
        folder_path=str(local_dir),
        repo_id=repo_id,
        repo_type='model',
        commit_message=f'Upload ARO Coder 4-bit ({SOURCE_LABEL})',
    )
    print(f'\n✓ Uploaded to https://huggingface.co/{repo_id}')

# Uncomment to push:
# push_to_hf(QUANT_DIR, HF_REPO_ID)

print('HuggingFace push ready.')
print(f'To publish, call: push_to_hf(QUANT_DIR, "{HF_REPO_ID}")')
print('(Make sure you ran: huggingface-cli login)')

## Step 5 — Release summary

In [ ]:
print('=' * 60)
print('ARO Coder — Release Summary')
print('=' * 60)
print(f'Source model:    {SOURCE_MODEL}')
print(f'Source label:    {SOURCE_LABEL}')
print(f'Base model:      {MODEL_ID}')
print()

if QUANT_DIR.exists():
    size_gb = sum(f.stat().st_size for f in QUANT_DIR.rglob('*') if f.is_file()) / 1e9
    print(f'Quantized model: {QUANT_DIR}')
    print(f'  Size:          {size_gb:.1f} GB')
print()

print('Distribution targets:')

if SERVE_SCRIPT.exists():
    print(f'  [✓] MLX server  →  {SERVE_SCRIPT}')
    print(f'      Start:  bash {SERVE_SCRIPT}')
    print(f'      API:    http://localhost:8080/v1/chat/completions')
print()

if MODELFILE_PATH.exists():
    ollama_ok = shutil.which('ollama') is not None
    status = '✓' if ollama_ok else '○ (Ollama not installed)'
    print(f'  [{status}] Ollama      →  {MODELFILE_PATH}')
    print(f'      Build:  ollama create {OLLAMA_NAME} -f {MODELFILE_PATH}')
    print(f'      Run:    ollama run {OLLAMA_NAME}')
print()

print(f'  [○] HuggingFace →  https://huggingface.co/{HF_REPO_ID}')
print(f'      Publish: push_to_hf(QUANT_DIR, "{HF_REPO_ID}")')
print()